Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
#@title Environment Setup

from pathlib import Path
import os

OPTIONS = {}

WORKSPACE = 'ComfyUI'
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"

    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo "-= Initial setup ComfyUI (Original)=-" && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo "-= Updating ComfyUI =-"
  # Descartamos cambios locales para evitar errores al hacer git pull
  !git reset --hard
  !git clean -fd
  !git pull
  !rm -f "/content/drive/MyDrive/ComfyUI/custom_nodes/comfyui-impact-pack.py"

%cd custom_nodes
if not os.path.exists('ComfyUI-Impact-Pack'):
  !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack
else:
  !echo "-= Updating ComfyUI-Impact-Pack =-"
  %cd ComfyUI-Impact-Pack
  !git reset --hard
  !git pull
  %cd ..

%cd $WORKSPACE

!echo -= Install dependencies =-
!pip -q install xformers -r requirements.txt

Mounting Google Drive...
/
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
-= Updating ComfyUI =-
Updating files: 100% (807/807), done.
HEAD is now at 24de8dc0 Fix SolidMask and MaskComposite device mismatch with --gpu-only (#13296)
Removing =0.13.3
Removing =0.4.2
Removing =0.7.1
Removing =4.28.1
Removing cloudflared-linux-amd64.deb
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 11 (delta 9), reused 11 (delta 9), pack-reused 0 (from 0)


### Run ComfyUI with localtunnel (Recommended Way)




In [3]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 971ms
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏setup plugin alembic.autogenerate.schemas
setup plugin alembic.autogenerate.tables
setup plugin alembic.autogenerate.types
setup plugin alembic.autogenerate.constraints
setup plugin alembic.autogenerate.defaults
setup plugin alembic.autogenerate.comments
setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.13.15 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-03 04:50:03.194
** Platform: Linux
** Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/drive/MyDrive/ComfyUI
** ComfyUI Base Folder Path: /content/drive/MyDrive/ComfyUI
** User directory: /content/drive/MyDrive/ComfyUI/user
** ComfyUI-Manager config path: /content/drive/M

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server